# Customer Liability Coverage Reconciliation

This notebook walks through the portfolio demonstration: load a synthetic customer ledger, aggregate liabilities with DuckDB, inspect one public BTC wallet balance, and render a reconciliation report. The customer ledger is synthetic and the public wallet addresses are demo data, not any real exchange's reserves.

In [1]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append(str(Path('..').resolve()))
ledger_path = Path('../data/customer_ledger.csv')
ledger = pd.read_csv(ledger_path)
ledger.head()

  customer_id asset      balance       as_of
0    C-000001  USDC  2902.920000  2026-05-05
1    C-000002   ETH     2.164536  2026-05-05
2    C-000002  USDC   333.640000  2026-05-05
3    C-000003  USDC   737.320000  2026-05-05
4    C-000004   ETH     0.519630  2026-05-05

The ledger is long-form: one row per customer-asset balance. A customer can hold multiple assets, which is why the synthetic 10,000-customer population creates 18,000 ledger rows.

In [2]:
from src.ledger import aggregate_by_asset, load_liabilities

liabilities = aggregate_by_asset(load_liabilities(ledger_path)).reset_index()
liabilities

  asset total_balance  customer_count     top_1pct_share
0   BTC         140.0            4000        0.408702575
1   ETH        8400.0            5000  0.404596945852381
2  USDC    12400000.0            6000 0.4108278217741936
3  USDT     4200000.0            3000 0.41426023095238096

The top 1% share is intentionally high to mimic a retail exchange liability book with a fat tail.

In [3]:
from src.chain import fetch_btc_balance

btc_balance = fetch_btc_balance('1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa')
print(f'BTC genesis address balance: {btc_balance} BTC')

BTC genesis address balance: 57.19331671 BTC


The full CLI also calls Etherscan for ETH, USDC, and USDT. Those calls require `ETHERSCAN_API_KEY`; tests mock them so CI never depends on live APIs.

In [4]:
from decimal import Decimal

demo_recon = pd.DataFrame([
    {'asset': 'BTC', 'customer_liabilities': Decimal('140'), 'on_chain_reserves': Decimal('142'), 'coverage_ratio': '101.429%', 'status': 'OK', 'usd_value_at_risk': Decimal('0')},
    {'asset': 'ETH', 'customer_liabilities': Decimal('8400'), 'on_chain_reserves': Decimal('8300'), 'coverage_ratio': '98.810%', 'status': 'BREACH', 'usd_value_at_risk': Decimal('310000')},
    {'asset': 'USDC', 'customer_liabilities': Decimal('12400000'), 'on_chain_reserves': Decimal('12410000'), 'coverage_ratio': '100.081%', 'status': 'OK', 'usd_value_at_risk': Decimal('0')},
    {'asset': 'USDT', 'customer_liabilities': Decimal('4200000'), 'on_chain_reserves': Decimal('4179000'), 'coverage_ratio': '99.500%', 'status': 'WATCH', 'usd_value_at_risk': Decimal('21000')},
])
demo_recon

  asset customer_liabilities on_chain_reserves coverage_ratio  status usd_value_at_risk
0   BTC                  140               142       101.429%      OK                 0
1   ETH                 8400              8300        98.810%  BREACH            310000
2  USDC             12400000          12410000       100.081%      OK                 0
3  USDT              4200000           4179000        99.500%   WATCH             21000

In [5]:
'HTML report renderer writes the same fields to output/recon_YYYY-MM-DD.html.'

HTML report renderer writes the same fields to output/recon_YYYY-MM-DD.html.

In production, this would add controlled price sourcing, alerting, review sign-off, stronger wallet ownership evidence, and asset-specific reserve policies. For this portfolio demonstration, the useful signal is the control shape: source data, aggregate, compare reserves, flag exceptions, and preserve daily evidence.